# Nube por vano -- todos los circuitos

Cuaderno hermano de `03_uiti_vano_trayectorias_circuitos.ipynb` y
`04_uiti_vano_trayectorias_vano.ipynb`, con el mismo esquema: todo se precomputa en
Python y un panel HTML+JS maneja la figura sin recalcular nada.

A diferencia de la version anterior de este cuaderno, aqui los ~208 circuitos viajan
TODOS al navegador (como en 03): el selector de **circuito** del panel cambia el
circuito activo EN VIVO -- reescribe geometria, UITI, nube y violines de las mismas 14
trazas para el circuito elegido, sin recargar la pagina ni volver a correr el cuaderno.
Ya no existe el aviso de "este cuaderno arma un circuito por vez" de la version previa.

El mapa colorea cada vano por su UITI diario, en cortes de cuartil fijos PARA ESE
CIRCUITO (mismo estilo que 03/04). Encima va una "nube" translucida: un unico trazo
Scattermap de marcadores, uno por vano con datos ese dia, cuya intensidad (alpha del
color) codifica el valor de la variable elegida. El `<select>` de variable del panel
ofrece **las 6**, en dos grupos:

- **Climaticas (por rezago horario)** -- precipitacion, temperatura, rafaga y velocidad
  del viento. El slider de **horas antes del evento** recorre sus 25 rezagos (0 a 24).
- **Estaticas del vano (sin rezago)** -- riesgo por vegetacion (`NR_T`) y descargas a
  tierra (`DDT`). Son atributos del vano, no series: el slider de hora se deshabilita
  mientras una de ellas este activa.

Un tercer slider elige el **dia** (solo fechas con eventos del circuito activo). Tanto el
hover como el click sobre un punto de la nube muestran el vano, la variable activa y su
valor con unidad EN LA HORA VIGENTE -- ese valor se reconstruye cada vez que se mueve el
slider de hora o se cambia de variable, nunca queda una foto vieja.

Debajo del mapa van **6 violines en 2 filas x 3 columnas**, las mismas 6 variables, para
los vano-eventos del dia elegido y sincronizados al circuito y dia activos. El eje y de
cada violin rotula su **unidad de medida**.


In [1]:
# Descomentar solo si el entorno no tiene instaladas estas dependencias.
# %pip install pandas numpy plotly geopandas


In [ ]:
# El tablero ya no vive en estas celdas: vive en `src/chec_tableros/clima.py`, con
# pruebas propias y contra un golden que compara el HTML byte a byte. Este cuaderno lo
# LLAMA, para que haya UNA implementacion y no dos que se separan en silencio.
#
# Lo que se pierde respecto de antes: el panel ya no se pinta dentro de la celda. El
# modulo no corre en un kernel, asi que no hay nada que `display` pueda recibir. El
# documento que se abre en el navegador es el MISMO, y ademas usa todo el ancho de la
# pantalla en vez del de la celda.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from chec_tableros import clima

# ABRIR_EN_NAVEGADOR: ponlo en False para escribir el archivo sin abrir nada (Databricks,
# Colab, nbconvert). La aplicacion de escritorio no pasa por aqui: llama al modulo.
ABRIR_EN_NAVEGADOR = True

RUTA_PANEL = clima.construir(raiz=ROOT, abrir=ABRIR_EN_NAVEGADOR)
print(f'panel autocontenido en {RUTA_PANEL.relative_to(ROOT)} '
      f'({RUTA_PANEL.stat().st_size / 1024 ** 2:,.1f} MB)')


## Como leerlo

- **Cada circuito tiene su propio numero de dias con eventos**, entre 1 y 79 (mediana 14).
  La etiqueta del slider lo dice, y en los 12 circuitos que solo registran un dia el slider
  **se deshabilita**: no hay nada que recorrer, y la serie de tiempo dibuja ese unico punto.
  No es un fallo del tablero, es lo que hay en la base para ese circuito.
- El **circuito** se elige en vivo desde el panel: el `<select>` de circuito lista los
  ~208 disponibles y, al cambiarlo, reescribe geometria, UITI, nube y violines de las
  mismas 14 trazas para el circuito elegido -- sin recargar la pagina ni volver a correr
  el cuaderno. Los dias del slider tambien se actualizan al circuito activo.
- La **variable de la nube** tambien se elige en vivo, entre LAS 6 y sin volver a
  ejecutar Python. `VARIABLE_CLIMA` solo decide con cual arranca el panel. La nube sigue
  siendo LA MISMA traza -- cambiar de variable solo repinta sus puntos (color y hover),
  igual que mover el slider de hora. El `<select>` las separa en dos grupos porque se
  comportan distinto:
  - **Climaticas**: una serie de 25 rezagos por vano; el slider de hora las recorre.
  - **Estaticas del vano** (`NR_T`, `DDT`): un unico valor por vano. Viajan con largo 1
    en vez de repetir 25 veces el mismo numero (eso inflaria el JSON del panel ~50%), el
    JS las indexa con clamp y el slider de hora **se deshabilita** mientras esten
    activas, en vez de quedar mintiendo que mueve algo.
- El **mapa** colorea cada vano por su UITI acumulado del dia elegido, en 4 cuartiles con
  cortes fijos PARA EL CIRCUITO ACTIVO (mismo criterio que 03/04, sobre dias en vez
  de ventanas mensuales). Esa capa va al **doble de ancho** (7 px) y **opaca**, encima de la
  estructura negra de 1,5 px: el color que se ve en el mapa es exactamente el de la muestra
  que el panel imprime en su leyenda, no una mezcla con el fondo. Un vano **sin eventos ese dia, o que no aparece en la
  lista del dia, no recibe NADA de esta capa**: queda solo su linea negra de estructura.
- La **nube** es un unico trazo de puntos translucidos, uno por vano con datos ese dia,
  ubicado en su centroide, del MISMO color que el violin de esa variable. Tanto el hover
  como el panel de **click** muestran el vano, la variable activa y su valor con unidad
  en la hora vigente (o "atributo estatico del vano" si la variable no tiene rezago).
- El valor de cada circulo se codifica con el **COLOR**, no con la opacidad. La
  **opacidad es constante en 0.5** para todos los puntos (`NUBE_OPACIDAD`), lo justo para
  que la red se siga leyendo por debajo de circulos de 78 px que se solapan entre vanos
  vecinos.
  - El **tono base del violin** de cada variable y el de **su serie** en (1,3) salen de
    esa MISMA escala (`TONO_POR_VAR`, la rampa muestreada en 0.72), no de una paleta
    aparte: los tres elementos de una variable hablan el mismo idioma cromatico.
  - Cada variable tiene su **escala secuencial** (`ESCALA_POR_VAR`): Precipitacion
    `Blues`, Temperatura `OrRd`, Rafaga de viento `BuGn`, Velocidad del viento `BuGn`,
    Riesgo por vegetacion `Greens`, Descargas a tierra `Oranges`. Las dos variables de
    viento comparten escala a proposito (asi se pidieron), asi que en el mapa se ven
    iguales; se distinguen por el `<select>` y por el color de sus violines, que si son
    distintos.
  - `cmin`/`cmax` se fijan sobre el **dataset completo** (`RANGO_GLOBAL`, los 208
    circuitos), no sobre el circuito activo: un color significa el mismo valor aunque se
    cambie de circuito. Contrapartida asumida: un circuito con poca variacion en esa
    variable ocupa solo un tramo corto de su escala.
  - El color lo resuelve **Plotly**, no el JS: `marker.color` lleva los valores crudos y
    la escala viaja en el mismo `restyle`, para que un cambio de variable nunca pinte un
    instante los valores nuevos con la escala vieja.
  - La tira "Escala de la nube" del panel muestra 5 muestras de la escala activa, con la
    misma opacidad que el mapa para no prometer un color mas saturado del que se ve.
- El slider de **horas antes del evento** va de 0 (la hora del evento) a 24; mover solo
  este slider repinta la nube (color + hover) sin tocar el mapa ni los violines.
- La **grilla** es de **4 filas x 3 columnas**: el **mapa** ocupa un bloque 2x2 en las
  posiciones (1,1), (1,2), (2,1) y (2,2); en **(1,3)** va la serie de tiempo de doble eje;
  la posicion **(2,3) queda vacia** a proposito; y los **6 violines** llenan las tres
  columnas de las filas 3 y 4.
- La **serie de tiempo** de (1,3) cruza dos cosas sobre los dias con eventos del circuito:
  - Eje **izquierdo** (rojo oscuro, continuo): el **UITI total del circuito ese dia**, o
    sea la suma sobre todos sus vanos. Es un total POR DIA, no una suma corrida: sube y
    baja, y por eso se puede leer contra la otra serie.
  - Eje **derecho** (color de la variable activa, punteado): la **mediana diaria de la
    variable elegida**, calculada sobre los MISMOS valores por vano-evento que alimentan
    su violin. No depende del slider de hora a proposito -- una serie por dias no deberia
    saltar al mover un slider que no le compete. El rotulo, el color y la unidad de este
    eje cambian con el `<select>` de variable.
  - Al mover el slider de **dia**, el punto de ese dia se pinta al **triple** de tamaño
    (9 -> 27 px) en AMBAS series, para ubicar de un vistazo donde esta el resto del panel.
  - Los dias sin eventos no existen en el eje: la serie solo tiene los dias del circuito.
- Todo el texto -- panel y figura -- va al **doble** del tamaño previo, para leerse en
  pantalla grande y a pantalla completa en el navegador. Las cuatro constantes
  (`FUENTE_BASE`, `FUENTE_SUBTITULO`, `FUENTE_EJE_TITULO`, `FUENTE_TITULO`) son la unica
  fuente de verdad de la figura; el CSS del panel replica esos mismos valores.
- Los **violines** muestran, para el circuito y dia elegidos, la distribucion entre los
  vano-eventos de ese dia:
  - Fila 1 -- **Precipitacion** (mm), **Temperatura** (°C) y **Rafaga de viento**
    (km/h): cada evento aporta su propio promedio de 12h (`_0`..`_11`), no un promedio
    por vano.
  - Fila 2 -- **Velocidad del viento** (km/h, misma media de 12h), **Riesgo por
    vegetacion** `NR_T` (indice) y **Descargas a tierra** `DDT` (descargas/km²/año):
    estas dos ultimas son atributos ESTATICOS del vano, asi que cada evento aporta el
    valor del vano donde ocurrio -- la distribucion refleja que vanos fallaron ese dia,
    no una variacion temporal.
  - Cada panel tiene su propio eje y, rotulado con la **unidad de medida** de su
    variable, y se actualiza junto con el circuito y el dia.